# Week 1 · Day 5 — Subqueries
**Goal:** Write queries inside queries to solve problems that can't be answered in a single pass.

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

**Day 4 gaps to fix today:**
- AVG grain: `AVG(o.total_amount)` is still order-level — correct pattern is inner query computes item revenue per order, outer query averages per segment
- LEFT JOIN to find zero-match records: `WHERE status <> 'completed'` is wrong — needs filtered subquery join then IS NULL
- Never skip comparison/proof drills

---
## Business First Prompt

> *Your manager asks: "I want to know which customers are spending above average, which regions have more revenue than the company average, and which products are being bought by our top-spending segments — but only for last quarter. Can you pull that?"*

Write 3–5 sentences below before touching any data:
- Why can't you answer this with a single flat query?
- What do you need to compute first before you can filter?
- What does a subquery let you do that WHERE alone cannot?

**Your answer:**

> I can't answer these questions in a single flat query because the filters depend on aggregated values that don't exist yet when WHERE runs. WHERE executes before GROUP BY, so there's no average to compare against until after aggregation is complete. I need to compute the company average revenue first, in a subquery, so the outer query can use that value as a filter. A subquery lets me run an aggregation independently and pass the result to the outer query as a condition, which a plain WHERE clause cannot do on its own.

---
## Setup — run this first

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('week1_ecommerce.db')
print('Connected to week1_ecommerce.db')

Connected to week1_ecommerce.db


---
## Table Preview — run this so you know what you're working with

In [6]:
for table in ['orders', 'customers', 'order_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- orders ---
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', 'region']


,order_id,customer_id,order_date,status,total_amount,region
0,1001,1,2024-01-05,completed,250.00,West
1,1002,2,2024-01-12,cancelled,89.99,East
2,1003,3,2024-01-20,completed,430.50,West



--- customers ---
Columns: ['customer_id', 'name', 'email', 'signup_date', 'segment', 'country']


,customer_id,name,email,signup_date,segment,country
0,1,Alice Martin,alice@email.com,2022-03-15,VIP,US
1,2,Bob Chen,bob@email.com,2022-07-22,Regular,CA
2,3,Sara Lopez,sara@email.com,2023-01-10,Regular,US



--- order_items ---
Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


---
## Concept 1 — What Is a Subquery?

A subquery is a query nested inside another query. The inner query runs first and produces a result that the outer query uses.

**Three places a subquery can live:**

| Location | Syntax | Use case |
|---|---|---|
| In WHERE | `WHERE col > (SELECT AVG(col) FROM ...)` | Filter rows against a computed value |
| In FROM | `FROM (SELECT ... ) sub` | Treat a query result like a table |
| In SELECT | `(SELECT ... WHERE match)` | Add a computed column per row |

**Why you need them:**  
SQL executes in one pass — you can't use an aggregate result in a WHERE clause directly.  
This fails:
```sql
-- WRONG — can't use WHERE on an aggregate
SELECT name FROM customers WHERE total_spend > AVG(total_spend)
```
This works:
```sql
-- RIGHT — compute the average first, then filter
SELECT name FROM customers
WHERE total_spend > (SELECT AVG(total_spend) FROM customers)
```

In [7]:
# EXAMPLE — orders above the average order value
q = """
SELECT order_id, total_amount, region, status
FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders WHERE status = 'completed')
  AND status = 'completed'
ORDER BY total_amount DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,order_id,total_amount,region,status
0,1027,980.0,West,completed
1,1018,890.0,East,completed
2,1023,720.0,West,completed
3,1030,640.0,West,completed
4,1007,620.0,South,completed
5,1021,560.0,South,completed
6,1010,510.0,North,completed
7,1013,480.0,West,completed


In [8]:
for table in ['orders', 'customers', 'order_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- orders ---
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', 'region']


,order_id,customer_id,order_date,status,total_amount,region
0,1001,1,2024-01-05,completed,250.00,West
1,1002,2,2024-01-12,cancelled,89.99,East
2,1003,3,2024-01-20,completed,430.50,West



--- customers ---
Columns: ['customer_id', 'name', 'email', 'signup_date', 'segment', 'country']


,customer_id,name,email,signup_date,segment,country
0,1,Alice Martin,alice@email.com,2022-03-15,VIP,US
1,2,Bob Chen,bob@email.com,2022-07-22,Regular,CA
2,3,Sara Lopez,sara@email.com,2023-01-10,Regular,US



--- order_items ---
Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


---
## Drill 1 — Customers who spent above the average customer spend

Find customers whose total completed order value is above the average total completed order value across all customers.  
Columns: name, segment, total_spend.  

Hint: inner query computes `AVG(total_amount)` across all completed orders. Outer query filters customers whose sum exceeds it.

In [9]:
q1 = """ SELECT c.name,
                 c.segment, 
                SUM(o.total_amount) AS total_spend
        FROM customers c join orders o on c.customer_id = o.customer_id
        WHERE o.status = 'completed' AND
                total_amount > (SELECT AVG(total_amount)
                                FROM orders
                                WHERE status = 'completed') 
        GROUP BY c.name, c.segment
        ORDER BY total_spend DESC

"""
df = pd.read_sql_query(q1, conn)
display(df)

,name,segment,total_spend
0,Carlos Mendez,VIP,1870.0
1,Omar Hassan,Regular,1200.0
2,James Wu,VIP,1070.0
3,Sara Lopez,Regular,640.0
4,Bob Chen,Regular,620.0


---
## Drill 2 — Proof drill: does the average match?

Run TWO queries:
- Query A: the average completed order value across all orders
- Query B: the minimum total_spend among customers returned in Drill 1

Query B's minimum should be above Query A's average. Confirm it. Don't skip this.

In [10]:
# Query A — overall average
q2a = """  SELECT ROUND(AVG(order_total), 2)
            FROM (SELECT o.order_id,
                        SUM(oi.quantity * oi.unit_price) AS order_total
                  FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
                  WHERE o.status = 'completed'
                  GROUP BY o.order_id)
"""
df = pd.read_sql_query(q2a, conn)
display(df)

,"ROUND(AVG(order_total), 2)"
0,269.29


In [11]:
# Query B — minimum spend among above-average customers
q2b = """ SELECT ROUND(MIN(total_spend), 2) AS min_above_avg_spend
            FROM (SELECT c.name,
                        AVG(o.total_amount) AS total_spend
                    FROM customers c JOIN orders o ON c.customer_id = o.customer_id
                    WHERE o.status = 'completed'
                    AND o.total_amount > (SELECT AVG(order_total)
                                        FROM (SELECT o.order_id,
                                                     SUM(oi.quantity * oi.unit_price) AS order_total
                                              FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
                                              WHERE o.status = 'completed'
                                              GROUP BY o.order_id)) 
                    GROUP BY c.name
                    )

"""
df = pd.read_sql_query(q2b, conn)
display(df)

,min_above_avg_spend
0,275.0


---
## Concept 2 — Subquery in FROM (Derived Table)

When you put a subquery in FROM, the result becomes a temporary table you can query against.  
This is the correct pattern for **grain separation** — the Day 4 AVG gap.

```sql
-- Wrong: AVG(o.total_amount) is order-level, not item-level
SELECT c.segment, AVG(o.total_amount) AS avg_order_value
FROM orders o JOIN customers c ON ...
GROUP BY c.segment

-- Right: compute item revenue per order first, then average per segment
SELECT segment, ROUND(AVG(order_revenue), 2) AS avg_order_value
FROM (
    SELECT c.segment,
           o.order_id,
           SUM(oi.quantity * oi.unit_price) AS order_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.segment, o.order_id
) sub
GROUP BY segment
```

**The inner GROUP BY collapses to order grain. The outer GROUP BY collapses to segment grain.**  
Two steps, two levels. That's grain separation.

In [12]:
# EXAMPLE — correct AVG order value per segment using grain separation
q = """
SELECT segment, ROUND(AVG(order_revenue), 2) AS avg_order_value
FROM (
    SELECT c.segment,
           o.order_id,
           SUM(oi.quantity * oi.unit_price) AS order_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.segment, o.order_id
) sub
GROUP BY segment
ORDER BY avg_order_value DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,segment,avg_order_value
0,VIP,359.43
1,Regular,234.61
2,New,163.00


---
## Drill 3 — AVG item-level order value per segment (grain separation)

Using the derived table pattern above: compute the average order value per customer segment at item grain.  
Columns: segment, avg_order_value (rounded to 2 decimal places).  

Inner query: `SUM(oi.quantity * oi.unit_price)` per `(segment, order_id)`  
Outer query: `AVG(order_revenue)` per segment

In [13]:
q3 = """ SELECT segment,
                ROUND(AVG(order_revenue), 2) AS avg_order_value
        FROM (SELECT c.segment,
                    o.order_id,
                    SUM(oi.quantity * oi.unit_price) AS order_revenue
                    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
                    JOIN customers c ON o.customer_id = c.customer_id
                    WHERE o.status = 'completed'
                    GROUP BY c.segment, o.order_id
        )sub
        GROUP BY segment
        ORDER BY avg_order_value DESC

"""
df = pd.read_sql_query(q3, conn)
display(df)

,segment,avg_order_value
0,VIP,359.43
1,Regular,234.61
2,New,163.00


---
## Drill 4 — Proof drill: compare AVG(o.total_amount) vs subquery AVG

Run both versions for VIP segment and show the numbers side by side.  
- Query A: `AVG(o.total_amount)` joined to customers, filtered to VIP  
- Query B: subquery grain separation for VIP only  

Do they differ? Why or why not? Write your answer in the markdown cell below.

In [14]:
# Query A — order-level AVG for VIP
q4a = """   SELECT c.segment,
                        SUM(o.total_amount) AS order_revenue
                    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
                    JOIN customers c ON o.customer_id = c.customer_id
                    WHERE o.status = 'completed' AND c.segment = 'VIP'
                    GROUP BY c.segment
                                

"""
df = pd.read_sql_query(q4a, conn)
display(df)

,segment,order_revenue
0,VIP,7160.0


In [15]:
# Query B — item-level AVG for VIP using subquery
q4b = """ SELECT segment, ROUND(AVG(item_revenue), 2) AS avg_item_value
            FROM ( SELECT c.segment,
                        o.order_id,
                        SUM(oi.quantity * oi.unit_price) AS item_revenue
                    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
                    JOIN customers c ON o.customer_id = c.customer_id
                    WHERE o.status = 'completed' AND c.segment = 'VIP'
                    GROUP BY c.segment, o.order_id
            ) 

"""
df = pd.read_sql_query(q4b, conn)
display(df)

,segment,avg_item_value
0,VIP,359.43


**What did you observe? Do the numbers differ and why?**

> Yes they differ, query A represents the average of total orders placed by the the vip segment. Query B represents average per item revenue.

---
## Drill 5 — Regions above the company average revenue

Show only regions where total completed revenue exceeds the average regional revenue.  
Columns: region, total_revenue.  

Hint: inner query computes average revenue per region, outer query filters regions above that.

In [16]:
q5 = """ SELECT region, ROUND((total_amount_spent), 2) AS total_revenue
            FROM ( SELECT o.region,
                            SUM(o.total_amount) AS total_amount_spent
                    FROM orders o
                    WHERE o.status = 'completed' 
                    GROUP BY o.region
                    HAVING total_amount_spent > (SELECT AVG(reg_total)
                                        FROM (SELECT o.region,
                                                     SUM(o.total_amount) AS reg_total
                                              FROM orders o 
                                              WHERE o.status = 'completed'
                                        GROUP BY o.region))

                
            )sub
            ORDER BY total_revenue DESC

"""

df = pd.read_sql_query(q5, conn)
display(df)

,region,total_revenue
0,West,3690.5


---
## Concept 3 — Subquery in WHERE with IN

So far every subquery returned **one value** — an average, a max. `IN` is for when the inner query returns a **list of values** and you want to check whether each outer row belongs to that list.

**How each query sees the world:**
- The inner query runs first, completely on its own. It has no knowledge of the outer query.
- The outer query runs second. It can only see what its own joins and the inner result give it.
- They share nothing except the result the inner query passes up.

**Which tables to join where:**
- Inner query: join only the tables you need to build the list
- Outer query: join the tables you need to return the final columns

```sql
-- Find customers who ordered in regions with at least one order over $500
SELECT c.name, c.segment, o.region
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id   -- outer needs name + region
WHERE o.region IN (
    SELECT DISTINCT region                         -- inner builds the list
    FROM orders                                    -- inner only needs orders
    WHERE total_amount > 500
      AND status = 'completed'
)
AND o.status = 'completed'
GROUP BY c.name, c.segment, o.region
ORDER BY o.region
```

**Why the joins are different in each layer:**
- The inner query only needs `region` — it only touches `orders`
- The outer query needs `name` and `segment` — it joins `customers` too
- Each layer joins exactly what it needs and nothing more

**NOT IN works the same way in reverse** — show everything NOT in the list.


In [17]:
# EXAMPLE — customers in regions with at least one order over $500
q = """
SELECT c.name, c.segment, o.region
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.region IN (
    SELECT DISTINCT region
    FROM orders
    WHERE total_amount > 500
      AND status = 'completed'
)
AND o.status = 'completed'
GROUP BY c.name, c.segment, o.region
ORDER BY o.region
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,segment,region
0,Alice Martin,VIP,East
1,Carlos Mendez,VIP,East
2,Bob Chen,Regular,North
3,James Wu,VIP,North
4,Lily Thompson,New,North
5,Bob Chen,Regular,South
6,David Kim,New,South
7,Emma Rossi,Regular,South
8,James Wu,VIP,South
9,Priya Nair,New,South


---
## Drill 6 — Customers in above-average revenue regions

Find customers who placed completed orders in regions where total completed revenue is above the average regional revenue.  
Columns: name, segment, region.

**Two questions to answer first:**
- ① Which regions have above-average revenue? → this is your inner query, returns a list of regions
- ② Which customers ordered in those regions? → this is your outer query

**What each layer joins:**
- Inner query needs region and revenue → join `orders` and `order_items`
- Outer query needs customer name, segment, region → join `customers` and `orders`

**What the inner query computes:**
- Total revenue per region using `SUM(oi.quantity * oi.unit_price)`
- Filter with `HAVING` where that total exceeds `AVG` of all regional totals
- Returns a list of region names

Hint: the average in HAVING is a scalar — `AVG(total_amount)` directly on the grouped orders table.


In [18]:
q6 = """
SELECT DISTINCT name
FROM (
    SELECT c.name,
           o.region,
           SUM(o.total_amount) AS total_amount
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.name, o.region
    HAVING SUM(o.total_amount) > (
        SELECT AVG(region_total)
        FROM (
            SELECT region,
                   SUM(total_amount) AS region_total
            FROM orders
            WHERE status = 'completed'
            GROUP BY region
        )
    )
)
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,segment,region
0,Alice Martin,VIP,East
1,Carlos Mendez,VIP,East
2,Bob Chen,Regular,North
3,James Wu,VIP,North
4,Lily Thompson,New,North
5,Bob Chen,Regular,South
6,David Kim,New,South
7,Emma Rossi,Regular,South
8,James Wu,VIP,South
9,Priya Nair,New,South


---
## Drill 7 — Products never bought by non-VIP customers

Show product names that appear exclusively in completed orders from VIP customers — they never appear in a completed order from any other segment.  
Columns: product_name.

**Two questions:**
- ① Which products appear in completed orders from non-VIP customers? → inner query, returns a list of product names
- ② Which products are NOT in that list? → outer query using NOT IN

Hint: the inner query joins `order_items`, `orders`, `customers` to find product names where segment != 'VIP'.  
The outer query finds product names from `order_items` that are NOT IN that list.


In [19]:
q7 = """
SELECT DISTINCT product_name
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
AND c.segment = 'VIP'
AND product_name NOT IN (
    SELECT DISTINCT product_name
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    AND c.segment != 'VIP'
)
"""
df = pd.read_sql_query(q7, conn)
display(df)

,product_name
0,Wireless Mouse
1,USB Hub
2,Desk Mat
3,Ergonomic Mouse
4,Wrist Rest
5,Keycap Set
6,Ultrawide Monitor
7,Monitor Arm
8,Ergonomic Chair
9,Curved Monitor


---
## Concept 4 — LEFT JOIN to Find Zero-Match Records

This is the Day 4 gap: `WHERE status <> 'completed'` finds customers who have non-completed orders — not customers with zero completed orders. These are different questions.

**The wrong approach:**
```sql
-- Finds customers who have at least one non-completed order
-- Does NOT find customers with zero completed orders
SELECT c.name FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status <> 'completed'
```

**The correct approach — filter before joining:**
```sql
-- Step 1: build a subquery of only completed orders
-- Step 2: LEFT JOIN to that — customers with no match get NULL
-- Step 3: IS NULL catches those with zero completed orders
SELECT c.name FROM customers c
LEFT JOIN (
    SELECT customer_id, order_id
    FROM orders
    WHERE status = 'completed'
) comp ON c.customer_id = comp.customer_id
WHERE comp.order_id IS NULL
```

**Why the subquery goes inside the JOIN:**
- The subquery pre-filters to completed orders before the join happens
- LEFT JOIN then finds customers with no match in that filtered set
- IS NULL on a column from the right side = no match found
- Without the subquery, the LEFT JOIN includes all orders and the filter breaks

**What each layer sees:**
- Inner query (inside JOIN): only the `orders` table, filtered to completed
- Outer query: `customers` table + the result of the inner query


In [20]:
# EXAMPLE — customers with zero completed orders
q = """
SELECT name, segment
FROM customers c
LEFT JOIN (
    SELECT customer_id
    FROM orders
    WHERE status = 'completed'
) comp ON c.customer_id = comp.customer_id
WHERE comp.customer_id IS NULL
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,segment


---
## Drill 8 — Customers with no completed orders

Using the pattern above: find customers who have never completed an order.  
Columns: customer_id, name, segment, country.

Use a subquery inside LEFT JOIN to pre-filter completed orders, then IS NULL.


In [21]:
q8 = """ SELECT c.customer_id,
                c.name,
                c.segment,
                c.country
            FROM customers c LEFT JOIN ( SELECT order_id,
                                                customer_id
                                            FROM orders
                                            WHERE status = 'completed'
                                         ) sub ON c.customer_id = sub.customer_id
            WHERE sub.customer_id IS NULL
"""
df = pd.read_sql_query(q8, conn)
display(df)

,customer_id,name,segment,country


In [22]:
for table in ['orders', 'customers', 'order_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- orders ---
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', 'region']


,order_id,customer_id,order_date,status,total_amount,region
0,1001,1,2024-01-05,completed,250.00,West
1,1002,2,2024-01-12,cancelled,89.99,East
2,1003,3,2024-01-20,completed,430.50,West



--- customers ---
Columns: ['customer_id', 'name', 'email', 'signup_date', 'segment', 'country']


,customer_id,name,email,signup_date,segment,country
0,1,Alice Martin,alice@email.com,2022-03-15,VIP,US
1,2,Bob Chen,bob@email.com,2022-07-22,Regular,CA
2,3,Sara Lopez,sara@email.com,2023-01-10,Regular,US



--- order_items ---
Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


---
## Drill 9 — Proof drill: wrong vs right LEFT JOIN

Run both versions and compare row counts:
- Query A: `WHERE o.status <> 'completed'` after LEFT JOIN (wrong approach)
- Query B: subquery pre-filter + IS NULL (correct approach)

How many rows does each return? Write below what the difference tells you.


In [23]:
# Query A — wrong approach
q9a = """SELECT c.name FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status <> 'completed'

"""
df = pd.read_sql_query(q9a, conn)
display(df)

,name
0,Bob Chen
1,James Wu
2,Priya Nair
3,Omar Hassan
4,Lily Thompson
5,Lily Thompson
6,Emma Rossi
7,Sara Lopez
8,Priya Nair
9,David Kim


In [24]:
# Query B — correct approach
q9b = """ SELECT c.customer_id,
                c.name,
                c.segment,
                c.country
            FROM customers c LEFT JOIN ( SELECT order_id,
                                                customer_id
                                            FROM orders
                                            WHERE status = 'completed'
                                         ) sub ON c.customer_id = sub.customer_id
            WHERE sub.customer_id IS NULL
"""

df = pd.read_sql_query(q9b, conn)
display(df)

,customer_id,name,segment,country


**What did each query return and why are they different?**

> The answer is completely different. First query tells me people that have had an order other than completed. Query 2 tells me what customer actually hasnt placed an order.


---
## Concept 5 — Correlated Subquery

All subqueries so far ran once and handed a fixed result to the outer query. A correlated subquery is different — it runs once **per row** of the outer query, referencing the current outer row each time.

**How it works:**
```sql
-- For each customer, find their single highest completed order
SELECT c.name,
       o.order_id,
       o.total_amount
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.total_amount = (
    SELECT MAX(total_amount)
    FROM orders
    WHERE customer_id = c.customer_id   -- references the CURRENT outer row
      AND status = 'completed'
)
AND o.status = 'completed'
```

**What each layer sees:**
- The inner query references `c.customer_id` from the outer query
- For customer A it asks "what is A's max order?" → filters to A's rows
- For customer B it asks "what is B's max order?" → filters to B's rows
- It runs once per customer row — the result changes each time

**When to use a correlated subquery:**
- You need a per-row computation that depends on the current outer row
- Finding the max/min/latest record per group
- Comparing each row to its own group's aggregate

**What each layer joins:**
- Outer query: joins the tables needed for the final output columns
- Inner query: only needs the table containing the value to compute — references the outer row via the WHERE clause


In [25]:
# EXAMPLE — each customer's highest single completed order
q = """
SELECT c.name,
       o.order_id,
       o.total_amount,
       o.region
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.total_amount = (
    SELECT MAX(total_amount)
    FROM orders
    WHERE customer_id = c.customer_id
      AND status = 'completed'
)
AND o.status = 'completed'
ORDER BY o.total_amount DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,order_id,total_amount,region
0,Carlos Mendez,1027,980.0,West
1,Omar Hassan,1023,720.0,West
2,Sara Lopez,1030,640.0,West
3,Bob Chen,1007,620.0,South
4,James Wu,1021,560.0,South
5,Emma Rossi,1026,415.0,South
6,Alice Martin,1015,390.0,East
7,Lily Thompson,1024,335.0,North
8,Priya Nair,1012,275.0,South
9,David Kim,1016,145.0,South


---
## Drill 10 — Each region's highest single completed order

For each region show the single highest completed order: order_id, total_amount, customer name.  
Use a correlated subquery that references `o.region` from the outer query.

**Two questions:**
- ① For this region, what is the highest order value? → correlated inner query, references outer region
- ② Show the order that matches that value → outer query filter

Hint: inner query: `SELECT MAX(total_amount) FROM orders WHERE region = o.region AND status = 'completed'`


In [26]:
q10 = """ SELECT c.name,
                o.region,
                 o.order_id,
                o.total_amount
                FROM customers c
                JOIN orders o ON c.customer_id = o.customer_id
                WHERE o.total_amount = ( SELECT MAX(total_amount)
                        FROM orders
                        WHERE region = o.region
                        AND status = 'completed'

                )
            
            AND o.status = 'completed'
            ORDER BY o.total_amount DESC


"""
df = pd.read_sql_query(q10, conn)
display(df)

,name,region,order_id,total_amount
0,Carlos Mendez,West,1027,980.0
1,Carlos Mendez,East,1018,890.0
2,Bob Chen,South,1007,620.0
3,James Wu,North,1010,510.0


---
## Drill 11 — Each customer's most recent completed order

For each customer show their most recent completed order: name, segment, order_id, order_date, total_amount.  
Use a correlated subquery to find MAX(order_date) per customer.

Hint: inner query references `c.customer_id` from the outer query.


In [27]:
q11 = """ SELECT c.name,
                c.segment,
                o.order_id,
                o.order_date,
                o.total_amount
                FROM customers c 
                JOIN orders o ON c.customer_id = o.customer_id
                WHERE o.order_date = ( SELECT MAX(order_date)
                                       FROM orders
                                       WHERE customer_id = c.customer_id
                                       AND status = 'completed'
                )
                AND o.status = 'completed'
                ORDER BY o.order_date DESC
"""
df = pd.read_sql_query(q11, conn)
display(df)

,name,segment,order_id,order_date,total_amount
0,Sara Lopez,Regular,1030,2024-12-28,640.0
1,Bob Chen,Regular,1029,2024-12-18,290.0
2,Carlos Mendez,VIP,1027,2024-12-02,980.0
3,Emma Rossi,Regular,1026,2024-11-25,415.0
4,Lily Thompson,New,1024,2024-11-03,335.0
5,Omar Hassan,Regular,1023,2024-10-22,720.0
6,James Wu,VIP,1021,2024-10-07,560.0
7,David Kim,New,1016,2024-08-05,145.0
8,Alice Martin,VIP,1015,2024-07-28,390.0
9,Priya Nair,New,1012,2024-06-08,275.0


---
## Drill 12 — Customers whose latest order was a cancellation

Find customers where the most recent order by date has status = 'cancelled'.  
Columns: name, segment, order_date, status.

Hint: correlated subquery finds MAX(order_date) per customer. Outer query then checks that the matching order has status = 'cancelled'.


In [28]:
q12 = """ SELECT c.name,
                c.segment,
                o.order_date,
                o.status
                FROM customers c 
                JOIN orders o ON c.customer_id = o.customer_id
                WHERE order_date = ( SELECT MAX(order_date)
                                        FROM orders
                                        WHERE customer_id = c.customer_id
                                        AND status = 'cancelled'
                                        )
                AND o.status = 'cancelled'
                ORDER BY o.order_date DESC

"""
df = pd.read_sql_query(q12, conn)
display(df)

,name,segment,order_date,status
0,Alice Martin,VIP,2024-12-10,cancelled
1,Priya Nair,New,2024-10-15,cancelled
2,Emma Rossi,Regular,2024-08-19,cancelled
3,Lily Thompson,New,2024-05-20,cancelled
4,Bob Chen,Regular,2024-01-12,cancelled


---
## Block 2 · 11:00am — Applied Challenge

> *Manager follow-up: "Based on yesterday's analysis, I want a full picture of our top-performing segment. Show me: which VIP customers are above the VIP average spend, what categories they're buying, and whether any of them have had a cancellation in the last quarter."*

Three queries, each using a different subquery pattern.

**Query 1:** VIP customers above the VIP average spend — subquery in WHERE/HAVING  
**Query 2:** Categories purchased by those above-average VIP customers — subquery with IN  
**Query 3:** Which of those VIP customers had a cancellation in Q4 2024 — correlated or IN subquery

Write your approach first.


**My approach — which subquery pattern fits each question and why:**

> *For query 1: VIP customers above average VIP spend. I will select vip customers and calculate the sum of total amount spent per order. The inner query will calculate the avg amount of all vip customer orders.
For query 2 : I will SELECT categories and us IN  to calculate above average vip customers
For query 3: I will select vip customer and us correlated query to select vip customer



In [29]:
# Query 1 — VIP customers above VIP average spend
q13 = """  SELECT c.name,

                SUM(o.total_amount) AS total_vip_spend
                FROM ( SELECT AVG (total_amount) AS avg_vip_spend
                        FROM customers c
                        JOIN orders o ON c.customer_id = o.customer_id
                        WHERE o.status = 'completed' AND c.segment = 'VIP' ) sub
                JOIN customers c ON c.customer_id = o.customer_id
                JOIN orders o ON c.customer_id = o.customer_id
                WHERE o.status = 'completed' AND c.segment = 'VIP'
                GROUP BY c.name
                HAVING SUM(o.total_amount) > sub.avg_vip_spend
                ORDER BY total_vip_spend DESC


                
"""
df = pd.read_sql_query(q13, conn)
display(df)

,name,total_vip_spend
0,Carlos Mendez,1870.0
1,James Wu,1070.0
2,Alice Martin,950.0


In [30]:
# Query 2 — categories bought by above-average VIP customers
q14 = """ SELECT c.name,
                oi.category
                FROM customers c
                JOIN orders o ON c.customer_id = o.customer_id
                JOIN order_items oi ON o.order_id = oi.order_id
                JOIN( SELECT AVG (o.total_amount) AS avg_vip_spend
                        FROM customers c
                        JOIN orders o ON c.customer_id = o.customer_id
                        WHERE o.status = 'completed' AND c.segment = 'VIP' ) sub
                WHERE o.status = 'completed' AND c.segment = 'VIP'
                GROUP BY c.name, oi.category
                HAVING SUM(o.total_amount) > sub.avg_vip_spend  
                ORDER BY c.name

            
                

                

"""
df = pd.read_sql_query(q14, conn)
display(df)

,name,category
0,Alice Martin,Accessories
1,Alice Martin,Electronics
2,Carlos Mendez,Accessories
3,Carlos Mendez,Electronics
4,James Wu,Furniture


In [31]:
# Query 3 — above-average VIP customers with a Q4 2024 cancellation
q15 = """ SELECT c.name
            FROM  customers c
            JOIN orders o ON c.customer_id = o.customer_id
            CROSS JOIN ( SELECT AVG(o.total_amount) AS avg_vip_spend
                    FROM customers c
                    JOIN orders o ON c.customer_id = o.customer_id
                    WHERE o.status = 'completed' AND c.segment = 'VIP'          
            ) sub
            WHERE o.status = 'cancelled'  AND c.segment = 'VIP'
            AND o.order_date >= '2024-10-01' AND o.order_date <= '2024-12-31'
            GROUP BY c.name
            HAVING SUM(o.total_amount) > sub.avg_vip_spend

"""
df = pd.read_sql_query(q15, conn)
display(df)

,name


**What does the data say? What would you tell your manager?**

> Carlos Mendez is a top spender, james vu and ALice Martin following up , accesories and electronics are tied for most bought and none have had a cancellation in the last quarter.


---
## Capstone — Full Segment Performance Subquery Report

Build a single query that shows for each segment:
- Total customers
- Completed orders
- Item-level revenue (`SUM(oi.quantity * oi.unit_price)`)
- Average order value at item grain (use grain separation — not `AVG(o.total_amount)`)
- Whether the segment is above or below the company-wide average item revenue per order — label it `'Above'` or `'Below'` using CASE WHEN

Order by item_revenue descending.

**Before writing SQL, answer:**
- What do you need to compute first?
- Which subquery pattern does each computed value need?
- Which tables does each layer need to join?


In [32]:
q_capstone = """ SELECT c.segment,
                        COUNT(DISTINCT c.customer_id)  AS total_customers,
                        COUNT(DISTINCT o.order_id)  AS total_completed_orders,
                        SUM(oi.quantity * oi.unit_price) AS item_revenue,
                        AVG(oi.quantity * oi.unit_price) AS avg_order_value,
                         CASE WHEN SUM(oi.quantity * oi.unit_price) > ( SELECT AVG(company_order_value)
                                                                         FROM ( SELECT o.order_id,
                                                                                SUM(oi.quantity * oi.unit_price) AS company_order_value
                                                                                 FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
                                                                                 WHERE o.status = 'completed'
                                                                                GROUP BY o.order_id)
                                                                                ) THEN 'Above' ELSE 'Below' END AS vs_company_value
                        FROM customers c JOIN orders o ON c.customer_id = o.customer_id 
                        JOIN order_items oi ON o.order_id = oi.order_id
                                              WHERE o.status = 'completed' 
                      GROUP BY c.segment
                      ORDER BY item_revenue DESC
                    

"""
df = pd.read_sql_query(q_capstone, conn)
display(df)

,segment,total_customers,total_completed_orders,item_revenue,avg_order_value,vs_company_value
0,VIP,3,7,2516.0,193.538462,Above
1,Regular,4,9,2111.5,124.205882,Above
2,New,3,3,489.0,163.000000,Above


In [33]:
q = """
SELECT c.segment, COUNT(DISTINCT o.order_id) AS completed_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.segment
"""
df = pd.read_sql_query(q, conn)
display(df)

,segment,completed_orders
0,New,3
1,Regular,9
2,VIP,7


In [34]:
q = """
SELECT c.segment, COUNT(DISTINCT o.order_id) AS completed_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN (
    SELECT order_id,
           SUM(quantity * unit_price) AS order_value
    FROM order_items
    GROUP BY order_id
) sub ON o.order_id = sub.order_id
WHERE o.status = 'completed'
GROUP BY c.segment
"""
df = pd.read_sql_query(q, conn)
display(df)

,segment,completed_orders
0,New,3
1,Regular,9
2,VIP,7


---
## Day 5 Checkpoint — answer before Day 6

Plain English — no SQL needed.


**1. What are the three places a subquery can live? Give a real use case for each.**

> A sub query can live in FROM, WHERE and in SELECT. The three are: WHERE — filter rows using a calculated value e.g. customers who spent more than the average. FROM — treat a subquery as a temporary table e.g. pre-aggregate order totals before joining to customers.SELECT — compute a single value per row e.g. pull the top category per segment inline 


**2. Why can't you write `WHERE total_spend > AVG(total_spend)` directly?**

> WHERE filters individual rows before any aggregation happens, so there's no aggregate value to compare against yet. Aggregates only exist after GROUP BY.


**3. What is grain separation and when do you need it?**

> Grain is the level of detail a row represents. Grain separation means keeping different levels of aggregation in separate subqueries rather than mixing them in one join. You need it when two aggregations happen at different levels for example, total revenue per customer (customer grain) and total revenue per category (category grain). If you try to compute both in one query with one GROUP BY, the results collide and inflate. You separate them into subqueries, then join the results together.


**4. What is the difference between a correlated subquery and a regular subquery?**

> A regular subquery runs once independently and returns a fixed result. A correlated subquery runs once per row of the outer query because it references a column from the outer query, it can't execute on its own.


**5. When you write a subquery with IN, what must the inner query return — and what does the outer query do with it?**

> The inner query must return a single column. The outer query checks each row to see if its value exists in that list, and keeps only the rows where it does.


**6. Why do the inner and outer queries sometimes join different tables?**

> The inner and outer queries have different purposes — the inner query often needs to reach a table the outer query doesn't need, in order to compute a value or filter a list. Each query only joins what it needs for its own job.